<a href="https://colab.research.google.com/github/AlbertoOliveros777/Programacion-Tarea/blob/main/Taller_Graficos_Publicacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller · De los datos a un dashboard publicado

**Programación — ADMI 1301**

Trabajarás con las **2.000 ventas de una cadena de papelerías en 2025**. En este taller vas a:

1. **Graficar** (6 ejercicios con `groupby`, `crosstab` y `pivot_table`).
2. **Organizar** tus gráficos en una sola figura con `subplots`: tu *dashboard*.
3. **Embellecerlo con prompts**: le pides a Gemini que reescriba el código de tu figura y pegas su respuesta aquí.
4. **Elegir tu versión favorita**, escribir tus conclusiones y **publicar tu notebook en GitHub**.

Las instrucciones completas están en la guía en Word. Aquí encuentras lo esencial de cada ejercicio y una **verificación** (`assert`) para saber si vas bien.

**Escriba los nombres de los integrantes aca.**
______________________________________________________________________  
 1.   Alberto Oliveros
 2.   Federico Gonzalez
 3.   Rafael Vargas
    


## 0. Preparación
Ejecuta la celda y sube el archivo **`ventas_papelerias_2025.csv`**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display

subidos = files.upload()   # selecciona ventas_papelerias_2025.csv

df = pd.read_csv("ventas_papelerias_2025.csv")
df["fecha"] = pd.to_datetime(df["fecha"])
df["mes"] = df["fecha"].dt.month
df.head()

**Conoce tus datos** antes de graficar: cuántas filas hay, qué columnas, qué valores tiene cada categoría.

In [ ]:
print(df.shape)
df.info()
for col in ["ciudad", "canal", "segmento", "categoria", "metodo_pago"]:
    print(col, "->", sorted(df[col].unique()))

---
# Parte A · Seis gráficos
En cada ejercicio: **(1)** prepara una tabla con `groupby`, `crosstab` o `pivot_table`, **(2)** guárdala en la variable indicada, **(3)** grafícala. Ponle **título** a cada gráfico.

### Ejercicio 1 · Top 5 de productos *(barras horizontales)*
**Contexto.** La gerente de compras debe decidir en qué productos concentrar el inventario del próximo año.

**Tarea.** Calcula los ingresos (`total`) por `producto`, quédate con los **5 mayores** y grafícalos en barras horizontales. Guarda la tabla en **`top_productos`**.

*Herramientas:* `groupby`, `sum`, `sort_values`, `head`, `.plot.barh()`

In [ ]:
# Ejercicio 1
import pandas as pd
top_productos = df.groupby("producto")["total"].sum().sort_values(ascending=False).head(5)
top_productos.sort_values().plot.barh(title="Top 5 productos por ingresos")


In [ ]:
# ✅ Verificación
assert len(top_productos) == 5, "Debe haber exactamente 5 productos"
assert top_productos.index[0] == "Cuaderno", "El producto con más ingresos no coincide: revisa el ordenamiento"
assert abs(top_productos.iloc[0] - 20712345) < 1, "Revisa que estés sumando la columna total"
print("Ejercicio 1 correcto ✔")

### Ejercicio 2 · Ingresos mes a mes: Bogotá vs Medellín *(líneas)*
**Contexto.** La dirección regional quiere comparar la evolución de sus dos ciudades principales durante el año.

**Tarea.** **Selecciona** solo las ventas de Bogotá y Medellín (`isin`). Con una **tabla dinámica** pon los meses en las filas, las ciudades en las columnas y la suma de `total` como valores. Guárdala en **`ing_mes_ciudad`** y grafícala como líneas con marcadores.

*Herramientas:* filtrado con `isin`, `pivot_table`, `.plot.line(marker="o")`

In [ ]:
# Ejercicio 2
dos_ciudades = df[df["ciudad"].isin(["Bogotá", "Medellín"])]
ing_mes_ciudad = dos_ciudades.pivot_table(index="mes", columns="ciudad", values="total", aggfunc="sum")
ing_mes_ciudad.plot.line(marker="o", title="Ingresos mes a mes: Bogotá vs Medellín")

In [ ]:
# ✅ Verificación
assert ing_mes_ciudad.shape == (12, 2), f"Se esperaba una tabla de 12 meses x 2 ciudades, tienes {ing_mes_ciudad.shape}"
assert set(ing_mes_ciudad.columns) == {"Bogotá", "Medellín"}, "Las columnas deben ser las dos ciudades"
assert abs(ing_mes_ciudad["Bogotá"].sum() - 37847614) < 1, "El total de Bogotá no coincide"
print("Ejercicio 2 correcto ✔")

### Ejercicio 3 · ¿Por qué canal se vende más? *(gráfico de pie)*
**Contexto.** La empresa evalúa si invierte más en su tienda online o en la atención telefónica.

**Tarea.** Calcula los ingresos totales por `canal`, guárdalos en **`ing_canal`** y grafica la participación de cada canal con un gráfico de pie que muestre los **porcentajes con un decimal**.

*Herramientas:* `groupby`, `.plot.pie(autopct=...)`

In [ ]:
# Ejercicio 3
ing_canal = df.groupby("canal")["total"].sum()
ing_canal.plot.pie(autopct="%1.1f%%", title="Participación de ingresos por canal")

In [ ]:
# ✅ Verificación
assert abs(ing_canal.sum() - 118668012) < 1, "La suma de los canales debe ser el ingreso total"
assert abs(ing_canal["Telefónico"] / ing_canal.sum() - 0.520) < 0.01, "La participación del canal Telefónico no coincide"
print("Ejercicio 3 correcto ✔")

### Ejercicio 4 · ¿Cada segmento compra lo mismo? *(barras apiladas)*
**Contexto.** El área comercial sospecha que los clientes corporativos compran productos distintos a los minoristas.

**Tarea.** Con `pd.crosstab` cuenta las ventas por `segmento` (filas) y `categoria` (columnas), **normalizadas por fila** y expresadas en **porcentaje** (de 0 a 100). Guarda la tabla en **`tabla_seg_cat`** y grafícala en barras **apiladas**.

*Herramientas:* `pd.crosstab(..., normalize="index")`, `.plot.bar(stacked=True)`

In [ ]:
# Ejercicio 4
tabla_seg_cat = pd.crosstab(df["segmento"], df["categoria"], normalize="index") * 100
tabla_seg_cat.plot.bar(stacked=True, title="Categorías compradas por segmento (%)")

In [ ]:
# ✅ Verificación
assert tabla_seg_cat.shape == (3, 5), f"Se esperaba 3 segmentos x 5 categorías, tienes {tabla_seg_cat.shape}"
assert (abs(tabla_seg_cat.sum(axis=1) - 100) < 0.01).all(), "Cada fila debe sumar 100 (usa normalize y multiplica por 100)"
print("Ejercicio 4 correcto ✔")

### Ejercicio 5 · Productividad de los vendedores *(dispersión)*
**Contexto.** Recursos humanos revisa qué vendedores concretan más ventas y cuáles venden tickets más altos.

**Tarea.** Con `groupby` por `vendedor` y `ciudad`, calcula para cada vendedor el **número de ventas** (`n_ventas`) y el **valor promedio de sus ventas** (`ticket_prom`). Guarda la tabla en **`vend`** (con índice normal: usa `reset_index()`). Haz una dispersión (`n_ventas` en X, `ticket_prom` en Y) con un **color por ciudad** y leyenda.

*Herramientas:* `groupby().agg(nombre=("columna", "función"))`, `reset_index`, `plt.scatter` dentro de un `for`

In [ ]:
# Ejercicio 5
vend = df.groupby(["vendedor", "ciudad"]).agg(
    n_ventas=("id_venta", "count"),
    ticket_prom=("total", "mean")
).reset_index()

for ciudad, grupo in vend.groupby("ciudad"):
    plt.scatter(grupo["n_ventas"], grupo["ticket_prom"], label=ciudad)

plt.xlabel("Número de ventas")
plt.ylabel("Ticket promedio")
plt.title("Productividad de los vendedores")
plt.legend()

In [ ]:
# ✅ Verificación
assert vend.shape[0] == 12, f"Debe haber una fila por vendedor (12), tienes {vend.shape[0]}"
assert {"vendedor", "ciudad", "n_ventas", "ticket_prom"} <= set(vend.columns), "Faltan columnas: vendedor, ciudad, n_ventas, ticket_prom"
assert vend["n_ventas"].sum() == 2000, "La suma de n_ventas debe ser 2000"
print("Ejercicio 5 correcto ✔")

### Ejercicio 6 · ¿Qué tan grandes son las ventas de cada segmento? *(diagrama de cajas)*
**Contexto.** Para planear el segundo semestre, finanzas quiere ver cómo se distribuye el valor de las ventas por tipo de cliente.

**Tarea.** **Selecciona** solo las ventas del **segundo semestre** (`mes` de 7 a 12) y guárdalas en **`sem2`**. Haz un diagrama de cajas del `total` por `segmento`, **sin mostrar los valores atípicos** (`showfliers=False`).

*Herramientas:* filtro con `>=`, `.boxplot(column=..., by=...)`

In [ ]:
# Ejercicio 6
sem2 = df[df["mes"] >= 7]
sem2.boxplot(column="total", by="segmento", showfliers=False)

In [ ]:
# ✅ Verificación
assert len(sem2) == 1050, f"El segundo semestre debe tener 1050 ventas, tienes {len(sem2)}"
print("Ejercicio 6 correcto ✔")

---
# Parte B · Todo en una sola figura (`subplots`)
### Ejercicio 7 · Organiza tus 6 gráficos en una cuadrícula de 2 × 3
**Contexto.** La gerente quiere ver todo en **una sola imagen** para presentarla en la junta.

**Tarea.** Crea una figura con `fig, axs = plt.subplots(2, 3, figsize=(18, 10))` y dibuja cada gráfico en su posición pasando **`ax=axs[fila, columna]`**. Agrega un título general con `fig.suptitle(...)` y termina con `plt.tight_layout()`.

**Importante:** escribe el código en la celda que empieza con `%%writefile figura_base.py` (así el programa guarda tu código en un archivo, que usarás en la Parte C). Luego ejecuta la celda siguiente (`%run -i figura_base.py`) para ver la figura.

*Pistas:* `plot.barh(ax=axs[0, 0])` · `plot.pie(...)` también acepta `ax` · para el boxplot: `boxplot(column=..., by=..., ax=...)` · para la dispersión: `axs[1, 1].scatter(...)`

In [ ]:
%%writefile figura_base.py
# Ejercicio 7: código de la figura 2x3
# Empieza con:  fig, axs = plt.subplots(2, 3, figsize=(18, 10))
fig, axs = plt.subplots(2, 3, figsize=(18, 10))

top_productos.sort_values().plot.barh(ax=axs[0, 0], title="Top 5 productos")
ing_mes_ciudad.plot.line(marker="o", ax=axs[0, 1], title="Bogotá vs Medellín")
ing_canal.plot.pie(autopct="%1.1f%%", ax=axs[0, 2], title="Ingresos por canal")
tabla_seg_cat.plot.bar(stacked=True, ax=axs[1, 0], title="Categorías por segmento (%)")

for ciudad, grupo in vend.groupby("ciudad"):
    axs[1, 1].scatter(grupo["n_ventas"], grupo["ticket_prom"], label=ciudad)
axs[1, 1].set_title("Productividad de vendedores")
axs[1, 1].legend()

sem2.boxplot(column="total", by="segmento", ax=axs[1, 2], showfliers=False)

fig.suptitle("Dashboard de ventas 2025")
plt.tight_layout()

In [ ]:
%run -i figura_base.py

In [ ]:
# ✅ Verificación
assert len(fig.axes) >= 6, f"La figura debe tener 6 gráficos, tiene {len(fig.axes)}"
print("Ejercicio 7 correcto ✔")

---
# Parte C · Hazla ver bonita con prompts
Ahora **tú escribes el prompt** y se lo envías a Gemini (en [gemini.google.com](https://gemini.google.com) o la IA que prefieras). Gemini **reescribe el código** de tu figura para que se vea profesional, casi como un dashboard. Tú pegas ese código en el notebook y lo ejecutas.

**El ciclo, cada vez:**
1. Ejecuta la celda de herramientas: te muestra **tu código actual** y las **reglas técnicas**.
2. Escribe tu prompt (Ejercicios 8 y 9).
3. En el chat de la IA pega, en este orden: **tu prompt + tu código + las reglas**.
4. Copia **solo el código** de la respuesta (botón *copiar* del bloque) y pégalo en la celda `%%writefile figura_v1.py`. **No incluyas las líneas con \`\`\`** ni el texto que la IA escriba fuera del bloque.
5. Ejecuta la celda y luego `mostrar("v1")` para ver la figura.

### Herramientas *(ya están hechas)*
Ejecuta la celda: define la función `mostrar(nombre)` (dibuja la versión guardada en `figura_<nombre>.py`) e imprime lo que debes copiar a la IA.

In [ ]:
REGLAS_FIGURA = """

Reglas técnicas obligatorias:
- Usa solo pandas y matplotlib (y numpy si lo necesitas). No instales otras librerías.
- Los datos ya están cargados en el DataFrame `df` (columnas: id_venta, fecha, ciudad, canal, segmento, vendedor,
  producto, categoria, metodo_pago, cantidad, precio_unitario, descuento, total, mes). No cargues archivos.
- Conserva los MISMOS 6 gráficos y los mismos datos del código actual; cambia solo la presentación.
- La figura debe quedar en una variable llamada `fig` (2 filas x 3 columnas).
- NO uses plt.show() ni guardes archivos.
- Entrega solo UN bloque de código Python completo.
"""


def mostrar(nombre):
    """Dibuja la versión guardada en figura_<nombre>.py."""
    plt.close("all")
    codigo = open(f"figura_{nombre}.py", encoding="utf-8").read().replace("plt.show()", "")
    exec(codigo, globals())
    display(fig)
    plt.close(fig)


print("=== 1) COPIA ESTE CÓDIGO (tu figura actual) ===")
print(open("figura_base.py", encoding="utf-8").read())
print("\n=== 2) COPIA ESTAS REGLAS al final de tu prompt ===")
print(REGLAS_FIGURA)

### Ejercicio 8 · Tu primer prompt de estilo
Completa las secciones de `prompt_v1`. Sé **específico**: "usa una paleta de azules y naranja" funciona mejor que "hazlo bonito".

- **PÚBLICO Y OBJETIVO:** quién verá la figura y para qué.
- **PALETA Y TIPOGRAFÍA:** colores (por ciudad, por producto…), fondo, tamaños de letra.
- **TÍTULOS:** que cada gráfico tenga un título que diga la **conclusión** (por ejemplo, "Bogotá lidera las ventas"), no solo el nombre.
- **EJES Y NÚMEROS:** formatos con separador de miles o "$ 12 M", etiquetas de ejes claras.
- **LIMPIEZA VISUAL:** quitar bordes y marcos innecesarios, cuadrícula suave, valores sobre las barras.

Ejecuta la primera celda para imprimir tu prompt y cópialo a la IA (con tu código y las reglas). Luego pega el código que te devuelva en la segunda celda y muestra la figura con la tercera.

In [ ]:
prompt_v1 = """
PÚBLICO Y OBJETIVO: La figura la verá la gerencia de la cadena de papelerías en una junta, para decidir inventario, canales y foco comercial del próximo año. Debe leerse de un vistazo.

PALETA Y TIPOGRAFÍA: Usa una paleta de azules con un naranja de acento. Fondo blanco, títulos en negrita de buen tamaño y etiquetas legibles. Un color distinto por ciudad en el gráfico de dispersión.

TÍTULOS: Cada gráfico debe tener un título que diga la CONCLUSIÓN, no solo el nombre. Por ejemplo "Cuaderno lidera los ingresos" o "Bogotá supera a Medellín todo el año".

EJES Y NÚMEROS: Formatea los valores de dinero con separador de miles o como "$ 12 M". Etiquetas de ejes claras y sin saturar.

LIMPIEZA VISUAL: Quita los bordes superiores y derechos, pon una cuadrícula suave, y muestra el valor sobre las barras del top de productos.
"""


print(prompt_v1)    # cópialo en la IA, junto con tu código y las reglas

In [ ]:
%%writefile figura_v1.py
# Pega aquí SOLO el código que te devolvió la IA (sin las líneas ```)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

AZUL = "#1f4e79"
AZUL_CLARO = "#5b9bd5"
NARANJA = "#ed7d31"
AZULES = ["#1f4e79", "#2e6da4", "#5b9bd5", "#9dc3e6", "#c9e0f5"]
COLOR_CIUDAD = {"Barranquilla": "#1f4e79", "Bogotá": "#ed7d31", "Bucaramanga": "#5b9bd5", "Cali": "#9dc3e6", "Medellín": "#2e6da4"}

def pesos_millones(x, pos):
    return f"$ {x/1e6:.0f} M"

def pesos_miles(x, pos):
    return f"$ {x/1000:.0f}k"

plt.rcParams.update({"axes.titleweight": "bold", "axes.titlesize": 12, "figure.facecolor": "white"})
fig, axs = plt.subplots(2, 3, figsize=(18, 10))

tp = top_productos.sort_values()
axs[0,0].barh(tp.index, tp.values, color=AZUL)
axs[0,0].set_title("Cuaderno lidera los ingresos")
axs[0,0].xaxis.set_major_formatter(FuncFormatter(pesos_millones))
for i, v in enumerate(tp.values): axs[0,0].text(v, i, f" {v/1e6:.1f}M", va="center", fontsize=9)

axs[0,1].plot(ing_mes_ciudad.index, ing_mes_ciudad["Bogotá"], marker="o", color=NARANJA, label="Bogotá")
axs[0,1].plot(ing_mes_ciudad.index, ing_mes_ciudad["Medellín"], marker="o", color=AZUL, label="Medellín")
axs[0,1].set_title("Bogotá supera a Medellín casi todo el año")
axs[0,1].set_xlabel("Mes")
axs[0,1].yaxis.set_major_formatter(FuncFormatter(pesos_millones))
axs[0,1].legend()

colores_canal = [NARANJA if c=="Telefónico" else AZUL_CLARO if c=="Online" else AZUL for c in ing_canal.index]
axs[0,2].pie(ing_canal.values, labels=ing_canal.index, autopct="%1.1f%%", colors=colores_canal)
axs[0,2].set_title("El canal Telefónico domina las ventas")

tabla_seg_cat.plot.bar(stacked=True, ax=axs[1,0], color=AZULES)
axs[1,0].set_title("Cada segmento compra categorías distintas")
axs[1,0].set_ylabel("% de ventas")
axs[1,0].legend(fontsize=7, loc="lower right")

for ciudad, grupo in vend.groupby("ciudad"): axs[1,1].scatter(grupo["n_ventas"], grupo["ticket_prom"], label=ciudad, color=COLOR_CIUDAD.get(ciudad, AZUL), s=60)
axs[1,1].set_title("Más ventas no implica mayor ticket")
axs[1,1].set_xlabel("Número de ventas")
axs[1,1].set_ylabel("Ticket promedio")
axs[1,1].yaxis.set_major_formatter(FuncFormatter(pesos_miles))
axs[1,1].legend(fontsize=7)

sem2.boxplot(column="total", by="segmento", ax=axs[1,2], showfliers=False)
axs[1,2].set_title("Distribución del valor de venta por segmento")
axs[1,2].set_xlabel("Segmento")
axs[1,2].set_ylabel("Total")

for ax in axs.flat: ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False); ax.grid(True, alpha=0.3)

fig.suptitle("Dashboard de ventas 2025 — Cadena de papelerías", fontsize=16, fontweight="bold")
plt.tight_layout()

In [ ]:
mostrar("v1")

### Ejercicio 9 · Itera: mejora tu prompt
Mira la figura que obtuviste: ¿qué no te gusta? Agrega una sección **AJUSTES** con **al menos dos cambios concretos** (otra paleta, letras más grandes, una dona en lugar del pie, anotar el valor máximo del gráfico de líneas…). Es la **iteración de prompts** que vimos en clase. Repite el ciclo: copia `prompt_v2` a la IA (con tu código base y las reglas), pega el código en `figura_v2.py` y muestra la figura.

In [ ]:
prompt_v2 = prompt_v1 + """

AJUSTES: 1) Cambia el pie por una DONA. 2) En el gráfico de líneas, anota el valor del mes más alto de cada ciudad. 3) Sube el tamaño de letra de los títulos.
"""
print(prompt_v2)

In [ ]:
%%writefile figura_v2.py
# Pega aquí el código que te devolvió la IA para la versión 2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

AZUL = "#1f4e79"
AZUL_CLARO = "#5b9bd5"
NARANJA = "#ed7d31"
AZULES = ["#1f4e79", "#2e6da4", "#5b9bd5", "#9dc3e6", "#c9e0f5"]
COLOR_CIUDAD = {"Barranquilla": "#1f4e79", "Bogotá": "#ed7d31", "Bucaramanga": "#5b9bd5", "Cali": "#9dc3e6", "Medellín": "#2e6da4"}

def pesos_millones(x, pos):
    return f"$ {x/1e6:.0f} M"

def pesos_miles(x, pos):
    return f"$ {x/1000:.0f}k"

plt.rcParams.update({"axes.titleweight": "bold", "axes.titlesize": 14, "figure.facecolor": "white"})
fig, axs = plt.subplots(2, 3, figsize=(18, 10))

tp = top_productos.sort_values()
axs[0,0].barh(tp.index, tp.values, color=AZUL)
axs[0,0].set_title("Cuaderno lidera los ingresos")
axs[0,0].xaxis.set_major_formatter(FuncFormatter(pesos_millones))
for i, v in enumerate(tp.values): axs[0,0].text(v, i, f" {v/1e6:.1f}M", va="center", fontsize=9)

axs[0,1].plot(ing_mes_ciudad.index, ing_mes_ciudad["Bogotá"], marker="o", color=NARANJA, label="Bogotá")
axs[0,1].plot(ing_mes_ciudad.index, ing_mes_ciudad["Medellín"], marker="o", color=AZUL, label="Medellín")
axs[0,1].set_title("Bogotá supera a Medellín casi todo el año")
axs[0,1].set_xlabel("Mes")
axs[0,1].yaxis.set_major_formatter(FuncFormatter(pesos_millones))
mb = ing_mes_ciudad["Bogotá"].idxmax(); vb = ing_mes_ciudad["Bogotá"].max()
axs[0,1].annotate(f"máx {vb/1e6:.1f}M", (mb, vb), textcoords="offset points", xytext=(0,10), color=NARANJA, fontsize=8, ha="center")
mm = ing_mes_ciudad["Medellín"].idxmax(); vm = ing_mes_ciudad["Medellín"].max()
axs[0,1].annotate(f"máx {vm/1e6:.1f}M", (mm, vm), textcoords="offset points", xytext=(0,10), color=AZUL, fontsize=8, ha="center")
axs[0,1].legend()

colores_canal = [NARANJA if c=="Telefónico" else AZUL_CLARO if c=="Online" else AZUL for c in ing_canal.index]
axs[0,2].pie(ing_canal.values, labels=ing_canal.index, autopct="%1.1f%%", colors=colores_canal, wedgeprops={"width": 0.45})
axs[0,2].set_title("El canal Telefónico domina las ventas")

tabla_seg_cat.plot.bar(stacked=True, ax=axs[1,0], color=AZULES)
axs[1,0].set_title("Cada segmento compra categorías distintas")
axs[1,0].set_ylabel("% de ventas")
axs[1,0].legend(fontsize=7, loc="lower right")

for ciudad, grupo in vend.groupby("ciudad"): axs[1,1].scatter(grupo["n_ventas"], grupo["ticket_prom"], label=ciudad, color=COLOR_CIUDAD.get(ciudad, AZUL), s=60)
axs[1,1].set_title("Más ventas no implica mayor ticket")
axs[1,1].set_xlabel("Número de ventas")
axs[1,1].set_ylabel("Ticket promedio")
axs[1,1].yaxis.set_major_formatter(FuncFormatter(pesos_miles))
axs[1,1].legend(fontsize=7)

sem2.boxplot(column="total", by="segmento", ax=axs[1,2], showfliers=False)
axs[1,2].set_title("Distribución del valor de venta por segmento")
axs[1,2].set_xlabel("Segmento")
axs[1,2].set_ylabel("Total")

for ax in axs.flat: ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False); ax.grid(True, alpha=0.3)

fig.suptitle("Dashboard de ventas 2025 — Cadena de papelerías", fontsize=16, fontweight="bold")
plt.tight_layout()

In [ ]:
mostrar("v2")

### Ejercicio 10 *(opcional)* · Una tercera versión
Si quieres seguir mejorando, crea `prompt_v3` (parte de `prompt_v2`), repite el ciclo con `figura_v3.py` y muestra `mostrar("v3")`.

In [ ]:
prompt_v3 = prompt_v2 + """

AJUSTES FINALES: 1) En la dispersión, agrega una línea horizontal de referencia con el ticket promedio general. 2) Haz los puntos más grandes y con borde blanco. 3) Ponle un fondo gris muy claro a cada subgráfico.
"""
print(prompt_v3)

In [ ]:
# ✅ Verificación
import os
assert os.path.exists("figura_v1.py") and os.path.exists("figura_v2.py"), "Faltan figura_v1.py y figura_v2.py: pega el código de cada versión"
assert "ESCRIBE AQUÍ" not in prompt_v1 and prompt_v2 != prompt_v1 and "ESCRIBE AQUÍ" not in prompt_v2, "Completa tus prompts v1 y v2"
print("Parte C correcta ✔")

---
# Parte D · Elige tu favorita y publícala
### Ejercicio 11 · Tu dashboard final y tus conclusiones
Mira tus versiones y **elige la que más te guste**. Luego escribe **3 hallazgos**: frases cortas con **cifras reales** sacadas de tus tablas. Se mostrarán debajo de tu dashboard.

*Ejemplo de hallazgo:* "Bogotá genera el 32 % de los ingresos de la cadena."

In [ ]:
figura_elegida = "v2"      # cambia por "v1", "v2" o "v3"

hallazgos = [
    "Bogotá concentra el 31.9 % de los ingresos de la cadena, la ciudad líder.",
    "El canal Telefónico genera el 51.8 % de los ingresos, más que Tienda y Online juntos.",
    "El Cuaderno es el producto estrella con $20.7 millones en ingresos del año.",
]

In [ ]:
# ✅ Verificación
import os
assert os.path.exists(f"figura_{figura_elegida}.py"), f"No existe figura_{figura_elegida}.py: revisa figura_elegida"
assert len(hallazgos) == 3 and all(len(h) > 25 and "ESCRIBE AQUÍ" not in h for h in hallazgos), "Escribe tus 3 hallazgos con cifras"
print("Ejercicio 11 correcto ✔")

### Tu dashboard final
Ejecuta la celda: muestra la versión que elegiste y tus tres hallazgos. **Esta es la salida que se verá en GitHub.**

In [ ]:
mostrar(figura_elegida)
print("Hallazgos:")
for h in hallazgos:
    print("•", h)

### Ejercicio 12 · Publica tu notebook en GitHub
GitHub muestra los notebooks **con sus figuras**, así que tu notebook publicado es tu dashboard. Solo necesitas una cuenta gratuita en [github.com](https://github.com).

**Antes de publicar:**
- Ejecuta todo el notebook de arriba abajo (*Entorno de ejecución → Ejecutar todo*) para que las figuras queden guardadas en las salidas.
- **Nunca escribas tu API key dentro de una celda.** Los *Secrets* de Colab no se guardan en el notebook y por eso son seguros; el repositorio será público.

**Pasos:**
1. **Crea el repositorio:** botón **New** → nombre `mi-dashboard-ventas` → **Public** → *Create repository*.
2. **Guarda el notebook en GitHub.** Opción A: en Colab, *Archivo → Guardar una copia en GitHub*, autoriza, elige tu repositorio y acepta. Opción B: *Archivo → Descargar → .ipynb* y, en el repositorio, **Add file → Upload files** y arrastra el archivo.
3. **Abre el notebook en el repositorio** (haz clic en el archivo `.ipynb`): GitHub lo dibuja con tus figuras.
4. **Copia el enlace** de la barra del navegador. Debe verse así: `https://github.com/TU-USUARIO/mi-dashboard-ventas/blob/main/NOMBRE.ipynb`
5. **Compruébalo:** ábrelo en una ventana de incógnito. Si lo ves sin iniciar sesión, está público.

**Si no se ven las figuras:** vuelve a Colab, ejecuta todo y guarda de nuevo. Si GitHub dice *Unable to render code block*, recarga la página unos segundos después.

---
**Para entregar:** (1) el **enlace público** de tu notebook en GitHub, con tu dashboard final visible, todos los integrantes deben estar como colaboradores en el repositorio, y (2) tus prompts (`prompt_v1`, `prompt_v2` y, si la hiciste, `prompt_v3`) más **tres líneas** sobre qué tuviste que corregir de lo que hizo Gemini y cómo lo pediste (escríbelas en una celda de texto al final del notebook) (3) El notebook (ipynb) con los nombres de todos los integrantes.